# 03_01 — Model Tuning: Parâmetro `contamination`

**Pipeline:** `model_training` (configuração via `conf/base/parameters.yml`)  
**Foco:** Análise do impacto do parâmetro `contamination` na latência de detecção

---

## Contexto

O parâmetro `contamination` do Isolation Forest é o **principal lever de tuning**:
ele define a fração dos dados de treino que o modelo assume serem anomalias,
o que determina o limiar de decisão.

| `contamination` | Efeito |
|---|---|
| muito baixo (0.001) | Limiar alto → detecta só anomalias extremas (latência alta) |
| valor médio (0.06) | Ponto ótimo empírico |
| muito alto (0.15+) | Limiar baixo → falsos alarmes durante voo normal |

**Objetivo deste notebook:** varrer diferentes valores de `contamination` e medir:
- Latência de detecção
- Falsos positivos no período normal

## Imports e parâmetros

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from aeroespacial_2.pipelines.model_training.nodes import (
    create_windows,
    evaluate_model,
    select_top_features_rf,
    split_windows,
    train_isolation_forest,
)

PREPARED_FILE = "../data/03_primary/carbonZ_2018-07-18-15-53-31_1_engine_failure.csv"
WINDOW_SIZE = 20
N_TOP_FEATURES = 10
TRAIN_RATIO = 0.7
TARGET_COL = "target_fault"
TIMESTAMP_COL = "timestamp"

In [ ]:
# Carregar e preparar os dados (igual ao 03_00)
df = pd.read_csv(PREPARED_FILE)
features = [c for c in df.columns if c not in [TARGET_COL, TIMESTAMP_COL]]
top_features = select_top_features_rf(df, features, TARGET_COL, n_top=N_TOP_FEATURES)

X, y = create_windows(df, WINDOW_SIZE, top_features, TARGET_COL)
timestamps = df[TIMESTAMP_COL].values[WINDOW_SIZE:]

X_train, X_test, y_train, y_test = split_windows(X, y, train_ratio=TRAIN_RATIO)
split_idx = int(len(timestamps) * TRAIN_RATIO)
ts_test = timestamps[split_idx:]

print(f"Dados prontos | Train: {X_train.shape} | Test: {X_test.shape}")

## Modelo de referência (`contamination = 0.06`)

Este é o valor configurado em `conf/base/parameters.yml`.

In [ ]:
contamination = 0.06
model_ref = train_isolation_forest(X_train, contamination=contamination)
metrics_ref = evaluate_model(model_ref, X_test, y_test, ts_test)

print(f"contamination = {contamination}")
print(f"  Falha real:  {metrics_ref.get('real_fault_time_s', 'N/A'):.2f}s")
print(f"  Detecção:    {metrics_ref.get('predicted_fault_time_s', 'N/A'):.2f}s")
print(f"  Latência:    {metrics_ref.get('detection_latency_s', 'N/A'):.2f}s")

## Varredura de `contamination`

Treina o modelo com diferentes valores e registra a latência de detecção
e o número de falsos positivos no período pré-falha.

In [ ]:
contamination_values = [0.001, 0.01, 0.03, 0.06, 0.08, 0.10, 0.15]
results = []

real_fault_idx = np.where(y_test == 1)[0]
real_fault_time = ts_test[real_fault_idx[0]] if len(real_fault_idx) > 0 else None

for c in contamination_values:
    m = train_isolation_forest(X_train, contamination=c)
    raw = m.predict(X_test)
    preds = np.where(raw == -1, 1, 0)

    # Latência de detecção
    met = evaluate_model(m, X_test, y_test, ts_test)
    latency = met.get("detection_latency_s")

    # Falsos positivos: alertas ANTES da falha real
    pre_fault_mask = ts_test < real_fault_time if real_fault_time else np.zeros_like(ts_test, dtype=bool)
    fp_count = int(preds[pre_fault_mask].sum())

    results.append({
        "contamination": c,
        "latency_s": latency,
        "false_positives": fp_count,
    })
    print(f"  c={c:.3f} | latência={f'{latency:.2f}s' if latency is not None else 'nao detectou':>10} | FP={fp_count}")

df_results = pd.DataFrame(results)
df_results

## Visualização: Latência vs Falsos Positivos

In [ ]:
df_plot = df_results.dropna(subset=["latency_s"])

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

ax1.plot(df_plot["contamination"], df_plot["latency_s"], "o-", color="tab:blue", label="Latência (s)")
ax2.bar(df_plot["contamination"], df_plot["false_positives"], alpha=0.4, color="tab:red",
        width=0.005, label="Falsos Positivos")

ax1.set_xlabel("contamination")
ax1.set_ylabel("Latência de detecção (s)", color="tab:blue")
ax2.set_ylabel("Falsos Positivos", color="tab:red")
ax1.set_title("Trade-off: Latência vs Falsos Positivos por Contamination")
ax1.grid(True, alpha=0.3)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2)
plt.tight_layout()
plt.show()

## Visualização detalhada — modelo de referência

Anomaly score ao longo do tempo de teste para `contamination = 0.06`.

In [ ]:
scores_ref = model_ref.decision_function(X_test)
y_pred_ref = np.where(model_ref.predict(X_test) == -1, 1, 0)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(ts_test, y_test, color="red", alpha=0.6, linewidth=2, label="Falha real")
axes[0].plot(ts_test, y_pred_ref, color="blue", linestyle="--", linewidth=1, label="Detecção")
if real_fault_time:
    axes[0].axvline(real_fault_time, color="black", linewidth=1.5, linestyle=":", label="Onset da falha")
axes[0].set_ylabel("Estado")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_title(f"contamination={contamination}")

axes[1].plot(ts_test, scores_ref, color="purple", linewidth=0.8)
axes[1].axhline(0, color="red", linestyle="--", linewidth=1.5, label="Limiar")
if real_fault_time:
    axes[1].axvline(real_fault_time, color="black", linewidth=1.5, linestyle=":")
axes[1].set_ylabel("Anomaly Score")
axes[1].set_xlabel("Tempo (s)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_title("Score de anomalia ao longo do tempo")

plt.tight_layout()
plt.show()

## Conclusão

Para aplicar o valor otimizado no Kedro, basta editar `conf/base/parameters.yml`:

```yaml
model_training:
  contamination: 0.06  # ajuste aqui
```

E rodar:
```
kedro run --pipeline=model_training
```